# Process-Level Behavioral Analytics

---

**Role:** Defender (Blue Team) · **Difficulty:** Intermediate · **Time:** ~50 min

> Signature-based detection — matching known malware hashes — fails against fileless attacks, living-off-the-land techniques, and zero-days. Behavioral analytics detects *what a process does*, not what it is. A legitimate `cmd.exe` spawning a reverse shell looks suspicious not because of its hash, but because `cmd.exe` making outbound network connections is abnormal for that host.

---

### What you'll understand after this notebook

1. How to build a probabilistic behavioral baseline: for each process, what fraction of instances normally have network connections?
2. Why probability-based detection adapts to host-specific behavior (a database server always has network connections; Notepad never should)
3. How to extend the original script with richer features: CPU anomalies, unusual parent-child process relationships, and new process detection
4. Where this fits in the UEBA (User and Entity Behavior Analytics) landscape and tools like Splunk UEBA and Microsoft Sentinel

---

### Real-world context

The SolarWinds Sunburst malware stayed undetected for 9 months because it used legitimate SolarWinds processes. It was eventually caught by FireEye through behavioral analysis — specifically, a `SolarWinds.BusinessLayerHost.exe` process making unexpected HTTPS connections to external IPs. Behavioral analytics caught what signature scanning missed.

## 1. The Core Idea: Probabilistic Behavioral Profiling

The original script builds a baseline by asking: **for each process name, what fraction of running instances normally have network connections?**

```python
conn_counts = {
    'chrome.exe':    (connected=45, total=50),   # prob = 0.90 — normally connected
    'notepad.exe':   (connected=0,  total=30),   # prob = 0.00 — never connected
    'svchost.exe':   (connected=12, total=20),   # prob = 0.60 — sometimes connected
    'explorer.exe':  (connected=1,  total=50),   # prob = 0.02 — rarely connected
}
```

Then during monitoring, if `notepad.exe` opens a network connection, its probability of being connected is 0.00 — below the 0.50 threshold — so it fires an alert.

If `SolarWinds.BusinessLayerHost.exe` suddenly starts making external connections with probability 0.01, that's your alert.

**Why this is better than a blocklist:**
- Adapts to this specific host's behavior
- Catches legitimate tools used maliciously (living off the land)
- Catches new processes that have never been seen before
- Threshold is tunable by process type

In [ ]:
import psutil                 # process and system information
import time                   # for simulated monitoring loop
import json                   # for saving/loading baseline
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import numpy as np
from collections import defaultdict
from datetime import datetime

# Verify psutil can see processes on this system
print(f'Visible processes: {len(psutil.pids())}')
print(f'Platform: {psutil.MACOS if hasattr(psutil, "MACOS") else psutil.WINDOWS if hasattr(psutil, "WINDOWS") else "Linux"}')

## 2. Building the Behavioral Baseline

In [ ]:
def build_baseline():
    """
    Scans all running processes and builds a probabilistic baseline:
    for each process name, tracks (instances_with_connections, total_instances).
    
    Returns:
        dict: {process_name: (connected_count, total_count)}
    """
    conn_counts = {}
    errors = 0
    
    for pid in psutil.pids():
        try:
            proc = psutil.Process(pid)
            name = proc.name()
            
            # Check if this process instance has any network connections
            has_connections = int(len(proc.connections()) > 0)
            
            if name in conn_counts:
                connected, total = conn_counts[name]
                conn_counts[name] = (connected + has_connections, total + 1)
            else:
                conn_counts[name] = (has_connections, 1)
                
        except (psutil.NoSuchProcess, psutil.AccessDenied):
            # Processes can die between pid scan and access
            # AccessDenied on system processes — skip silently
            errors += 1
    
    return conn_counts, errors


print('Building behavioral baseline...')
baseline, skip_count = build_baseline()

print(f'Baseline built: {len(baseline)} unique process names')
print(f'Skipped (access denied / terminated): {skip_count}')
print()

# Show baseline probabilities
print(f'{'Process':<35} {'Connected':<12} {'Total':<8} {'P(network)'}')
print('─' * 65)
for name, (connected, total) in sorted(baseline.items(), key=lambda x: x[1][0]/x[1][1], reverse=True)[:20]:
    prob = connected / total
    bar = '█' * int(prob * 10) + '░' * (10 - int(prob * 10))
    print(f'{name[:34]:<35} {connected:<12} {total:<8} {bar} {prob:.2f}')

## 3. Live Monitoring Against the Baseline

In [ ]:
THRESHOLD = 0.5  # probability below this = anomalous

def check_connections(baseline, threshold=THRESHOLD):
    """
    Scans current processes and flags anomalies against the baseline.
    
    Anomaly conditions:
    1. Known process making connections at below-threshold probability
    2. Known process NOT making connections at below-threshold probability  
    3. NEW process (not in baseline) making any network connection
    
    Returns list of alert dicts.
    """
    alerts = []
    
    for pid in psutil.pids():
        try:
            proc = psutil.Process(pid)
            name = proc.name()
            connections = proc.connections()
            has_conns = len(connections) > 0
            
            if name in baseline:
                connected, total = baseline[name]
                
                if has_conns:
                    # Process is connected — is this normal?
                    prob_connected = connected / total
                    if prob_connected < threshold:
                        alerts.append({
                            'type': 'unexpected_connection',
                            'process': name,
                            'pid': pid,
                            'baseline_prob': prob_connected,
                            'connections': [
                                f'{c.laddr.ip}:{c.laddr.port} → {c.raddr.ip if c.raddr else "?"}:{c.raddr.port if c.raddr else "?"}'
                                for c in connections if c.raddr
                            ],
                            'severity': 'HIGH' if prob_connected == 0 else 'MEDIUM'
                        })
                else:
                    # Process is NOT connected — is this unusual?
                    prob_not_connected = 1 - (connected / total)
                    if prob_not_connected < threshold:
                        # This process is almost always connected but isn't now
                        # Could indicate it was killed or is misbehaving
                        alerts.append({
                            'type': 'expected_connection_missing',
                            'process': name,
                            'pid': pid,
                            'baseline_prob': connected / total,
                            'severity': 'LOW'
                        })
            else:
                # New process not in baseline — flag if it has connections
                if has_conns:
                    alerts.append({
                        'type': 'new_process_with_connection',
                        'process': name,
                        'pid': pid,
                        'baseline_prob': None,
                        'connections': [
                            f'{c.laddr.ip}:{c.laddr.port} → {c.raddr.ip if c.raddr else "?"}:{c.raddr.port if c.raddr else "?"}'
                            for c in connections if c.raddr
                        ],
                        'severity': 'MEDIUM'
                    })
        except (psutil.NoSuchProcess, psutil.AccessDenied):
            continue
    
    return alerts


print('Running behavioral check against baseline...')
print(f'Threshold: P < {THRESHOLD} → alert')
print()

alerts = check_connections(baseline)

if alerts:
    for alert in alerts:
        sev_color = {'HIGH': '🔴', 'MEDIUM': '🟡', 'LOW': '🔵'}
        icon = sev_color.get(alert['severity'], '⚪')
        print(f"{icon} [{alert['severity']}] {alert['type']}")
        print(f"   Process: {alert['process']} (PID {alert['pid']})")
        if alert.get('baseline_prob') is not None:
            print(f"   Baseline P(connected): {alert['baseline_prob']:.2f}")
        if alert.get('connections'):
            for conn in alert['connections'][:3]:
                print(f"   Connection: {conn}")
        print()
else:
    print('No anomalies detected in current process snapshot.')

## 4. Visualizing the Baseline Distribution

In [ ]:
# Compute probability for each process
probs = {
    name: connected / total
    for name, (connected, total) in baseline.items()
    if total >= 2  # filter single-instance processes
}

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(13, 5))
fig.patch.set_facecolor('#0F0F0F')

# Left: histogram of connection probabilities
ax1.set_facecolor('#1A1A1A')
prob_values = list(probs.values())
n, bins, patches = ax1.hist(prob_values, bins=20, color='#6C9BCF', edgecolor='#282828')

# Color the low-probability bars (the suspicious zone)
for patch, b in zip(patches, bins):
    if b < THRESHOLD:
        patch.set_facecolor('#6CCF9B')   # normal (rarely connected)
    else:
        patch.set_facecolor('#6C9BCF')   # normal (often connected)

ax1.axvline(x=THRESHOLD, color='#CF6C6C', linestyle='--', label=f'Alert threshold ({THRESHOLD})')
ax1.set_xlabel('P(network connection)', color='#7A7570')
ax1.set_ylabel('Number of process types', color='#7A7570')
ax1.set_title('Process Connection Probability Distribution', color='#EDEAE4', fontsize=11)
ax1.tick_params(colors='#7A7570')
ax1.legend(facecolor='#1A1A1A', edgecolor='#282828', labelcolor='#EDEAE4', fontsize=9)
for spine in ax1.spines.values():
    spine.set_edgecolor('#282828')

# Right: top processes by connection rate
ax2.set_facecolor('#1A1A1A')
top_procs = sorted(probs.items(), key=lambda x: x[1], reverse=True)[:15]
names  = [p[0][:20] for p in top_procs]
values = [p[1] for p in top_procs]
colors = ['#CF6C6C' if v > 0.8 else '#6C9BCF' for v in values]

ax2.barh(names, values, color=colors, edgecolor='#282828')
ax2.set_xlabel('P(network connection)', color='#7A7570')
ax2.set_title('Top 15 Processes by Network Activity', color='#EDEAE4', fontsize=11)
ax2.tick_params(colors='#7A7570')
for spine in ax2.spines.values():
    spine.set_edgecolor('#282828')

plt.tight_layout()
plt.savefig('../cyberlab/content/articles/23_behavioral_baseline.png', dpi=150, bbox_inches='tight')
plt.show()

## 5. Saving the Baseline

A baseline is useless if rebuilt every run. In production, you'd build it over 24-48 hours and save it, then load it for ongoing monitoring.

In [ ]:
import json
from pathlib import Path

def save_baseline(baseline, path='baseline.json'):
    """Serialize baseline to JSON for persistence across runs."""
    serializable = {
        name: {'connected': c, 'total': t, 'prob': c/t}
        for name, (c, t) in baseline.items()
    }
    with open(path, 'w') as f:
        json.dump({
            'created': datetime.now().isoformat(),
            'threshold': THRESHOLD,
            'processes': serializable
        }, f, indent=2)
    print(f'Baseline saved to {path} ({len(serializable)} processes)')


def load_baseline(path='baseline.json'):
    """Load a previously saved baseline."""
    with open(path) as f:
        data = json.load(f)
    baseline = {
        name: (entry['connected'], entry['total'])
        for name, entry in data['processes'].items()
    }
    print(f'Loaded baseline from {data["created"]} ({len(baseline)} processes)')
    return baseline


save_baseline(baseline, '/tmp/cyberlab_baseline.json')

## 6. Limitations and Real-World Extensions

The original script is a proof-of-concept. Production UEBA adds:

| Feature | Why it matters |
|---------|----------------|
| **Parent-child process tracking** | `word.exe` spawning `cmd.exe` is a classic malware pattern |
| **Destination IP reputation** | Connection to known-bad IPs (threat intel feeds) |
| **Time-of-day baseline** | `outlook.exe` at 3am is weird; `cron` is normal |
| **CPU spike correlation** | Sudden CPU + new network connection = crypto miner / beacon |
| **Command line argument logging** | `powershell.exe -enc <base64>` = encoded payload |
| **Rolling baseline** | Baseline decays over time; adapts to legitimate new software |

### Commercial equivalents

- **CrowdStrike Falcon** — ML-based process behavioral analysis
- **Microsoft Defender for Endpoint** — process tree anomaly detection
- **Splunk UEBA** — entity profiling at scale
- **Vectra AI** — network behavioral analytics

What this notebook built is the conceptual foundation of all of them.

## 7. Article Seed

---

**Suggested title:** *Building a Behavioral Anomaly Detector in Python — The Concept Behind CrowdStrike and Sentinel*

**Opening paragraph (edit this):**

> The SolarWinds attackers stayed hidden for 9 months. Their malware used legitimate software, had valid digital signatures, and matched no known-bad hash. What eventually caught them was behavioral analysis — a process making connections that didn't match its historical pattern. In this article, I'll show you how to build that exact concept from scratch in Python, using nothing but `psutil` and probability theory.

**Section headers:**
1. Why signature detection fails against living-off-the-land attacks
2. Probabilistic behavioral profiling: building a baseline with psutil
3. Anomaly detection: flagging processes that break their pattern
4. Extending to production: what CrowdStrike adds on top of this foundation

**Medium tags:** `cybersecurity, python, blue-team, threat-detection, ueba`

---

In [ ]:
# Self-check

# Q1: Why does behavioral detection catch attacks that signature detection misses?
behavioral_advantage = None  # describe the concept
assert 'behav' in behavioral_advantage.lower() or 'pattern' in behavioral_advantage.lower() or 'legitim' in behavioral_advantage.lower(), \
    'It detects what a process DOES, not what it IS — catches legitimate tools used maliciously'

# Q2: What psutil method returns a process's active network connections?
connections_method = None  # method name as string
assert connections_method == 'connections', 'proc.connections() returns a list of connection objects'

# Q3: What threshold below which a process having a network connection is flagged?
our_threshold = None  # float
assert our_threshold == 0.5, 'P(connected) < 0.5 is considered anomalous in this implementation'

print('All checks passed. You understand process behavioral analytics.')